# Conformal Triage — Figure 1: the scope of a stratified guarantee in α

Regenerates `fig_alpha_scope.pdf` (Figure 1 of the paper) from the released metadata alone.
No model output enters this figure. Runs on CPU in ~2 min.

- **Panel (a)**: $\mathrm{cov}_5(\alpha)$, the 5th percentile of $\mathrm{Beta}(n+1-l,\,l)$ with
  $l=\lfloor (n+1)\alpha \rfloor$ (Eq. 3), at each stratum's mean de-clustered calibration count.
- **Panel (b)**: $\Pr(\text{degenerate})(\alpha) = \Pr(n_{\text{cal}} < \lceil 1/\alpha\rceil - 1)$,
  estimated over patient-level draws under the pre-specified allocations (HIBA: calibration = 70%
  of patients; PAD: training = 40% stratified by phototype, calibration = 60% of the remainder),
  de-clustered per stratum.

Inputs: `conformal-triage/emb/hiba_isic_metadata.csv` and `pad_isic_metadata.csv` (the ISIC-archive
metadata the pipeline uses). With `DRAWS = 4000` and `SEED = 2026` the console check reproduces the
`Pr(degenerate)` column of Table 6 to within its Monte-Carlo half-width. Output:
`conformal-triage/results/fig_alpha_scope.pdf` (+ `.png`). Copy the PDF next to `main.tex`.

Note: `figures/fig_alpha_scope.py` in the repository reads the original Mendeley/HIBA CSV schemas;
this notebook uses the ISIC-archive schema and the pipeline's own `lesion_table`, so the strata are
defined exactly as in the audit.

In [ ]:
# 1) Setup
SEED, DRAWS = 2026, 4000
import os, numpy as np, pandas as pd
from scipy.stats import beta as Beta
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/conformal-triage'
except Exception:
    BASE = os.environ.get('CT_BASE', os.path.expanduser('~/conformal-triage'))
SMOKE = os.environ.get('CT_SMOKE') == '1'
RES = f'{BASE}/results'; os.makedirs(RES, exist_ok=True)
Hm = pd.read_csv(f'{BASE}/emb/hiba_isic_metadata.csv')
Pm = pd.read_csv(f'{BASE}/emb/pad_isic_metadata.csv')

In [ ]:
# 2) Lesion table and stratum membership (identical to the pipeline)
CLS = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
MAL = {'MEL','BCC','SCC','AK'}
ROMAN = {'I':1,'II':2,'III':3,'IV':4,'V':5,'VI':6}
HIBA_MAP = {'melanoma':'MEL','nevus':'NV','basal cell carcinoma':'BCC',
            'squamous cell carcinoma':'SCC','actinic keratosis':'AK',
            'seborrheic keratosis':'BKL','solar lentigo':'BKL',
            'lichenoid keratosis':'BKL','dermatofibroma':'DF','vascular lesion':'VASC'}
DX3_MAP = {'Melanoma, NOS':'MEL','Melanoma':'MEL','Nevus':'NV',
           'Basal cell carcinoma':'BCC','Squamous cell carcinoma, NOS':'SCC',
           'Squamous cell carcinoma':'SCC','Solar or actinic keratosis':'AK',
           'Seborrheic keratosis':'BKL','Solar lentigo':'BKL',
           'Lichen planus like keratosis':'BKL','Lichenoid keratosis':'BKL',
           'Dermatofibroma':'DF','Hemangioma':'VASC','Angioma':'VASC',
           'Vascular lesion':'VASC',
           'Benign soft tissue proliferations - Vascular':'VASC'}

def lesion_table(meta, cohort):
    df = meta.copy()
    if 'diagnosis' in df.columns and df['diagnosis'].notna().any():
        dx = df['diagnosis']
    else:
        d2 = df['diagnosis_2'] if 'diagnosis_2' in df.columns else pd.Series(np.nan, index=df.index)
        dx = df['diagnosis_3'].fillna(d2)
    df['cls'] = dx.map({**HIBA_MAP, **DX3_MAP})
    df['g'] = df['fitzpatrick_skin_type'].map(ROMAN)
    missing = sorted(dx[df['cls'].isna()].dropna().unique().tolist())
    assert not missing, f'{cohort}: unmapped diagnoses: {missing}'
    les = df.groupby('lesion_id').agg(cls=('cls','first'), patient=('patient_id','first'), g=('g','first')).reset_index()
    les['mal'] = les.cls.isin(MAL)
    return les

def membership(les):
    m = {}
    for pid, sub in les[les.g.notna()].groupby('patient'):
        m[pid] = {('M', int(r.g)) if r.mal else ('B',) for _, r in sub.iterrows()}
    return m

hles, ples = lesion_table(Hm, 'hiba'), lesion_table(Pm, 'pad')
if not SMOKE:
    assert len(hles) == 1246 and len(ples) == 1891, (len(hles), len(ples))
hmem, pmem = membership(hles), membership(ples)
pf = ples.groupby('patient')['g'].agg(lambda s: tuple(s.dropna().unique()))
pgroup = {p: (int(v[0]) if len(v) else 0) for p, v in pf.items()}
print('lesions:', len(hles), 'HIBA |', len(ples), 'PAD')

In [ ]:
# 3) Patient-level draws under the pre-specified allocations (de-clustered counts per stratum)
rng = np.random.default_rng(SEED)
H_STRATA = [('M',1), ('M',2), ('M',3), ('B',)]
P_STRATA = [('M',1), ('M',2), ('M',3), ('M',4), ('B',)]

def counts_hiba(mem, strata, draws):
    pats = np.array(sorted(mem)); k = int(round(0.7 * len(pats)))
    out = {s: np.empty(draws, int) for s in strata}
    for i in range(draws):
        cal = set(rng.choice(pats, k, replace=False))
        for s in strata: out[s][i] = sum(1 for q in cal if s in mem[q])
    return out

def counts_pad(mem, grp, strata, draws):
    pats = np.array(sorted(grp)); g = np.array([grp[q] for q in pats])
    out = {s: np.empty(draws, int) for s in strata}
    for i in range(draws):
        tr = set()
        for gv in np.unique(g):
            idx = np.where(g == gv)[0]
            tr.update(pats[rng.choice(idx, int(round(0.4 * len(idx))), replace=False)])
        rem = np.array([q for q in pats if q not in tr])
        cal = set(rng.choice(rem, int(round(0.6 * len(rem))), replace=False))
        for s in strata: out[s][i] = sum(1 for q in cal if s in mem.get(q, ()))
    return out

hc = counts_hiba(hmem, H_STRATA, DRAWS)
pc = counts_pad(pmem, pgroup, P_STRATA, DRAWS)
print('mean de-clustered calibration counts:')
for name, c in [('HIBA', hc), ('PAD', pc)]:
    print(' ', name, {f"{s[0]}{s[1] if s[0]=='M' else ''}": round(float(v.mean()), 1) for s, v in c.items()})

In [ ]:
# 4) Analytic cov5 and empirical Pr(degenerate), and the figure
EPS = 1e-9
def cov5(alpha, n):
    l = int(np.floor((n + 1) * alpha + EPS))
    return np.nan if l < 1 else Beta.ppf(0.05, n + 1 - l, l)   # below alpha_min nothing is certified
def pr_deg(alpha, counts):
    return 100.0 * np.mean(counts < int(np.ceil(1.0 / alpha - EPS)) - 1)

ALPHAS = np.arange(0.02, 0.1601, 0.0002)
C = {'I': '#1f77b4', 'II': '#e08214', 'III': '#2ca02c', 'IV': '#d62728', 'B': '#7b52ab'}
curves = [
    ('HIBA mal. I & III', C['I'],   '-',  hc[('M',1)]),
    ('HIBA mal. II',      C['II'],  '-',  hc[('M',2)]),
    ('HIBA benign',       C['B'],   '-',  hc[('B',)]),
    ('PAD mal. I',        C['I'],   '--', pc[('M',1)]),
    ('PAD mal. II',       C['II'],  '--', pc[('M',2)]),
    ('PAD mal. III',      C['III'], '--', pc[('M',3)]),
    ('PAD mal. IV',       C['IV'],  '--', pc[('M',4)]),
    ('PAD benign',        C['B'],   '--', pc[('B',)]),
]
plt.rcParams.update({'font.size': 8.5, 'axes.labelsize': 9, 'legend.fontsize': 7.6,
                     'mathtext.fontset': 'cm', 'pdf.fonttype': 42,
                     'axes.spines.top': False, 'axes.spines.right': False})
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.0, 3.4))
for lab, col, ls, cnt in curves:
    n = int(round(cnt.mean()))
    ax1.plot(ALPHAS, [cov5(a, n) for a in ALPHAS], ls, color=col, lw=1.4, label=lab)
    ax2.plot(ALPHAS, [pr_deg(a, cnt) for a in ALPHAS], ls, color=col, lw=1.4)
for ax in (ax1, ax2):
    for a, lw in ((0.05, 0.9), (0.10, 0.5), (0.15, 0.5)):
        ax.axvline(a, color='0.75', lw=lw, ls=':', zorder=0)
    ax.set_xlim(0.02, 0.16); ax.set_xlabel(r'requested error level $\alpha$'); ax.set_xticks([0.02, 0.05, 0.10, 0.15])
# caption anchors on the 34-patient HIBA strata (n = 24)
ax1.plot([0.05, 0.10], [cov5(0.05, 24), cov5(0.10, 24)], 'o', color=C['I'], ms=3.5, zorder=5)
ax1.annotate(f'{cov5(0.05, 24):.3f}', (0.05, cov5(0.05, 24)), xytext=(3, -9), textcoords='offset points', fontsize=7.5, color=C['I'])
ax1.annotate(f'{cov5(0.10, 24):.3f}', (0.10, cov5(0.10, 24)), xytext=(5, -12), textcoords='offset points', fontsize=7.5, color=C['I'])
ax1.axhline(0.90, color='0.85', lw=0.6, zorder=0); ax1.set_ylim(0.60, 1.0)
ax1.set_ylabel(r'$\mathrm{cov}_5$ certified when a threshold exists')
ax1.text(0.02, 1.02, '(a)', transform=ax1.transAxes, fontsize=10, fontweight='bold')
ax2.set_ylim(-2, 102); ax2.set_ylabel(r'$\Pr(\mathrm{degenerate})$ [%]')
ax2.text(0.02, 1.02, '(b)', transform=ax2.transAxes, fontsize=10, fontweight='bold')
fig.legend(*ax1.get_legend_handles_labels(), loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.04))
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig(f'{RES}/fig_alpha_scope.pdf', bbox_inches='tight'); fig.savefig(f'{RES}/fig_alpha_scope.png', dpi=150, bbox_inches='tight')
print('figure ->', f'{RES}/fig_alpha_scope.pdf')

In [ ]:
# 5) Console check against Table 6 (Pr(degenerate) at 0.05 / 0.10 / 0.15) and the caption anchors
print(f'draws={DRAWS} seed={SEED}')
for lab, _, _, cnt in curves:
    print(f'{lab:18s} mean n_cal {cnt.mean():6.1f}  min {cnt.min():3d}  Pr(deg) .05/.10/.15 = '
          f'{pr_deg(0.05, cnt):5.2f} / {pr_deg(0.10, cnt):5.2f} / {pr_deg(0.15, cnt):5.2f} %')
print(f'caption anchors: cov5(0.05, n=24) = {cov5(0.05, 24):.3f}   cov5(0.10, n=24) = {cov5(0.10, 24):.3f}')
print('Table 6 reference: HIBA M1/M3 2.28/2.20 %, PAD B 0.22 %, PAD M4 100/36.9/2.9 %; anchors 0.883 / 0.817')